# Activation-Steering & Anti-Alignment — Experiment Console

One interactive environment over the **`refusal_geometry`** pipeline. Every stage is a single
cell that wraps the tested `asw` CLI/library — no need to read the repo. Edit the **Control
Panel** (Cell 3), then *Run All*.

**Pipeline:** download → extract `d_refuse` (C2) → anti-alignment map (C1) → fit condition (C4)
→ evaluate defenses → ablations → adversarial → **report** (regenerates every table & figure).

Each stage:
- is **idempotent** — re-running skips finished work (caches + prompt-level resume); set `FORCE=True` to recompute,
- streams **live logs** and shows **inline tables/plots**,
- writes artifacts under `/kaggle/working/refusal_geometry/{cache,data,results,report}` (persist across *Save Version*).

> **GPU:** uses `device_map="auto"`. ≤14B fits a single T4 (use `QUANT="int8"`/`"nf4"` for 14B);
> the 70B point is a paid A100 run, not this notebook.

## 1 · Clone / update the repository

In [ ]:
# Clones on first run, fast-forwards on later runs. Uses the GITHUB_TOKEN Kaggle secret.
import os
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "refusal_geometry"

if not REPO.exists():
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    os.system(f"git clone -q https://{token}@github.com/SLIMIHamda/refusal_geometry.git {REPO}")
else:
    os.system(f"git -C {REPO} pull --ff-only -q")

os.chdir(REPO)
print("cwd:", os.getcwd())
os.system("git -C . log --oneline -1")

## 2 · Environment

Kaggle ships `torch` (cu121) preinstalled — we keep it. We bump `transformers>=4.43` so
Qwen-2.5 loads, and add the CPU-side report deps. Then we confirm the GPU-free test spine is green.

In [ ]:
%pip install -q "transformers==4.44.2" "accelerate>=0.29.3" "bitsandbytes>=0.43.1" "pyyaml" "pandas" "pyarrow" "scipy" "scikit-learn" "matplotlib" "tabulate" "tqdm"
print("\nSpine tests (no GPU needed):")
!python -m pytest -q

## 3 · Control Panel — edit, then *Run All*

In [ ]:
# ============================ EXPERIMENT SETTINGS ============================
# Model under study — any file in configs/models/
MODEL_CONFIG = "configs/models/dolphin-llama3-8b.yaml"
QUANT        = None            # None | "int8" | "nf4"   (int8/nf4 for 14B on one T4)

# Geometry / steering band
STEER_LAYERS = None            # None = config's steer_layers; or e.g. [13, 14, 15, 16]
ALPHA        = 8.0             # steering strength for steered defenses

# Benchmarks
HARMFUL_BENCH    = "harmbench"     # harmful refusal eval
OVERREFUSE_BENCH = "xstest"        # benign / over-refusal eval
BENIGN_FOR_COND  = "orbench"       # benign negatives for the condition vector (over-refusal set)
PROMPT_LIMIT     = 100             # cap prompts/eval for speed; None = full split

# Defenses to evaluate (each is a Generator the harness scores identically)
DEFENSES = ["none", "system_prompt", "abliteration", "cast", "wrapper"]

# Ablation (alpha sweep)
ABLATE_ALPHAS = [2, 4, 8, 16]

# Statistics
SEEDS = [0]                    # [0, 1, 2] for the paper; 1 seed for a fast pass

# Stage toggles — switch off expensive stages while iterating
RUN_EXTRACT   = True
RUN_GEOMETRY  = True
RUN_CONDITION = True
RUN_DEFENSES  = True
RUN_ABLATION  = True
RUN_ATTACKS   = False          # Tier-1 (slow); demo only when True
RUN_REPORT    = True

FORCE = False                  # True = recompute even if a cache exists
DB    = "results/runs.sqlite"
# ============================================================================
print("Model :", MODEL_CONFIG, "| quant:", QUANT)
print("Eval  :", HARMFUL_BENCH, "/", OVERREFUSE_BENCH, "| limit:", PROMPT_LIMIT, "| seeds:", SEEDS)
print("Defs  :", DEFENSES)

## 4 · Helpers

Thin wrappers so each later cell is one call: `asw(...)` runs a CLI subcommand with live logging;
`load_runs(DB)` / the report tables turn the manifest into DataFrames; the `*_exists()` checks
drive idempotency by reusing the repo's own cache-path logic.

In [ ]:
import subprocess, time, sqlite3
import pandas as pd
from IPython.display import display, Markdown, Image
%matplotlib inline

from asw.config import load_config
from asw.harness.cli import _drefuse_path, _geometry_labels_path, _condition_path
from asw.report.load import load_runs
from asw.report import tables, figures


def sh(cmd, title=None):
    """Run a shell command, stream stdout, raise on non-zero exit."""
    if title:
        print(f"\n{'='*64}\n▶ {title}\n{'='*64}")
    print("$", cmd)
    t0 = time.time()
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout:
        print(p.stdout)
    if p.returncode != 0:
        print(p.stderr)
        raise RuntimeError(f"command failed ({p.returncode})")
    print(f"✓ {time.time()-t0:.1f}s")
    return p.stdout


def asw(subcmd, **flags):
    """Build and run `python -m asw.harness.cli <subcmd>` from keyword flags."""
    parts = [f"python -m asw.harness.cli {subcmd}"]
    for k, v in flags.items():
        if v is None or v is False:
            continue
        flag = "--" + k.replace("_", "-")
        if v is True:
            parts.append(flag)
        elif isinstance(v, (list, tuple)):
            parts.append(flag + " " + " ".join(str(x) for x in v))
        else:
            parts.append(f"{flag} {v}")
    return sh(" ".join(parts), title=subcmd)


def cfg():
    return load_config(MODEL_CONFIG)


def drefuse_exists():   return _drefuse_path(cfg()).exists()
def geometry_exists():  return _geometry_labels_path(cfg()).exists()
def condition_exists(): return _condition_path(cfg()).exists()


def show_runs(n=12):
    """Most recent runs from the manifest (every paper number traces back here)."""
    df = load_runs(DB)
    cols = [c for c in ["experiment", "model_id", "defense", "seed", "status", "started_at"] if c in df.columns]
    return df.sort_values("started_at", ascending=False)[cols].head(n) if not df.empty else df

print("helpers ready ·  d_refuse:", drefuse_exists(), "· geometry:", geometry_exists(), "· condition:", condition_exists())

## 5 · Download benchmarks → `data/*.jsonl`  (idempotent)

In [ ]:
from pathlib import Path

needed = sorted({HARMFUL_BENCH, OVERREFUSE_BENCH, BENIGN_FOR_COND, "advbench"})
missing = [b for b in needed if not Path(f"data/{b}.jsonl").exists()]
if FORCE or missing:
    sh(f"python scripts/download_benchmarks.py --benchmarks {' '.join(needed)} --limit 400",
       title="download benchmarks")
else:
    print("all benchmark jsonl present — skip")
sh("ls -la data/*.jsonl")

## 6 · Extract the refusal direction `d_refuse`  (C2)

Behavioral-contrast difference-in-means (same prompts, native vs forced-refusal) per band layer.
Cached to `cache/drefuse/<model>.npz`; layer-consistency cosines are logged as a sanity check.

In [ ]:
if RUN_EXTRACT and (FORCE or not drefuse_exists()):
    asw("extract", config=MODEL_CONFIG, layers=STEER_LAYERS, quant=QUANT)
elif RUN_EXTRACT:
    print("d_refuse cache present — skip (FORCE=True to recompute)")
else:
    print("RUN_EXTRACT=False — skipped")

## 7 · Anti-alignment map  (C1)

Projects activations onto `d_refuse` per layer and classifies the geometry by 95% CI sign.
**Anti-aligned (`<y,d> < 0`) is the headline** — the property naive steering assumes away.
Heatmap + table shown inline.

In [ ]:
if RUN_GEOMETRY and (FORCE or not geometry_exists()):
    asw("geometry-map", config=MODEL_CONFIG, quant=QUANT)
elif RUN_GEOMETRY:
    print("geometry labels present — skip (FORCE=True to recompute)")

if RUN_GEOMETRY:
    g = tables.table_geometry(load_runs(DB))
    display(g)
    Path("report/figures").mkdir(parents=True, exist_ok=True)
    p = figures.fig_anti_alignment_map(g, "report/figures/anti_alignment_map.png")
    if p:
        display(Image(p))

## 8 · Fit the harmful-input condition vector  (C4)

A CAST-style detector (harmful vs benign difference-in-means at the mid-band layer) that gates
the wrapper, so benign prompts pass through untouched. Train separation accuracy is logged.

In [ ]:
if RUN_CONDITION and (FORCE or not condition_exists()):
    asw("fit-condition", config=MODEL_CONFIG, benign=BENIGN_FOR_COND, quant=QUANT)
elif RUN_CONDITION:
    print("condition vector present — skip (FORCE=True to recompute)")
else:
    print("RUN_CONDITION=False — skipped")

## 9 · Evaluate defenses

Each defense is scored on the **harmful** and **over-refusal** benchmarks over `SEEDS`. Runs
resume at prompt granularity, so re-execution only fills gaps. Pooled refusal-rate table inline.

In [ ]:
if RUN_DEFENSES:
    from tqdm.auto import tqdm
    grid = [(d, b) for d in DEFENSES for b in (HARMFUL_BENCH, OVERREFUSE_BENCH)]
    for d, b in tqdm(grid, desc="defense × benchmark"):
        asw("eval", config=MODEL_CONFIG, benchmark=b, defense=d, alpha=ALPHA,
            quant=QUANT, limit=PROMPT_LIMIT, seeds=SEEDS)
    display(tables.table_refusal(load_runs(DB)))
else:
    print("RUN_DEFENSES=False — skipped")

## 10 · Ablation — steering strength `α`

Sweeps the wrapper's `α` on the harmful benchmark (safety/utility trade-off). The
layer-band and condition-on/off axes are available via `axis="layers"` / `"condition"`.

In [ ]:
if RUN_ABLATION:
    asw("ablate", config=MODEL_CONFIG, benchmark=HARMFUL_BENCH, axis="alpha",
        alphas=ABLATE_ALPHAS, quant=QUANT, limit=PROMPT_LIMIT, seeds=SEEDS)
    a = tables.table_ablation(load_runs(DB), "alpha")
    display(a)
    p = figures.fig_alpha_tradeoff(a, "report/figures/alpha_tradeoff.png")
    if p:
        display(Image(p))
else:
    print("RUN_ABLATION=False — skipped")

## 11 · Adversarial robustness  (C5) — demo

Attacks run against any Generator. Below is a **cheap multi-turn persona** demo comparing the
undefended model vs the wrapper. GCG (`asw.attacks.gcg.run_gcg`) and PAIR
(`asw.attacks.pair.run_pair`) are Tier-1 (slow / need an attacker LLM) — keep `RUN_ATTACKS=False`
unless you have the budget.

In [ ]:
if RUN_ATTACKS:
    from tqdm.auto import tqdm
    from asw.models.loader import load_model
    from asw.harness.cli import _build_generator
    from asw.scorers.judge import RubricJudge
    from asw.attacks.multiturn import run_multiturn
    from asw.attacks.common import attack_success_rate
    from asw.data.benchmarks import load_benchmark

    c = cfg()
    model, tok = load_model(c, quant=QUANT)
    judge = RubricJudge()
    behaviors = load_benchmark("advbench", data_dir="data").prompts()[:5]

    for label, defense in [("undefended", "none"), ("wrapper", "wrapper")]:
        gen = _build_generator(c, model, tok, defense, ALPHA)
        res = [run_multiturn(gen, b, judge, persona="dan") for b in tqdm(behaviors, desc=label)]
        print(f"{label:11s} multi-turn ASR = {attack_success_rate(res):.2f}  (n={len(res)})")
else:
    print("RUN_ATTACKS=False — skipped (set True for the multi-turn demo)")

## 12 · Report — regenerate every table & figure

`asw report` reads `runs.sqlite` and writes `report/REPORT.md` + `report/tables/*.csv` +
`report/figures/*.png`. Nothing in the paper is hand-transcribed.

In [ ]:
if RUN_REPORT:
    asw("report", config=MODEL_CONFIG, out="report")
    display(Markdown(Path("report/REPORT.md").read_text(encoding="utf-8")))
    for png in sorted(Path("report/figures").glob("*.png")):
        display(Image(str(png)))
else:
    print("RUN_REPORT=False — skipped")

## 13 · Run manifest (provenance)

In [ ]:
display(show_runs(20))

---
### Persist & resume
`cache/`, `data/`, `results/`, `report/` live under `/kaggle/working`, so **Save Version**
keeps them. On the next run, Cell 1 fast-forwards the repo and every stage resumes from its
cache — re-running is cheap. Flip a `RUN_*` toggle off to skip a stage, or `FORCE=True` to
recompute one. To study another model, point `MODEL_CONFIG` at a different `configs/models/*.yaml`.